In [10]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
import json
import time


def scrape_laws(start_page=1, max_pages=25, output_file="laws_luatVN.json"):
    all_data = []
    current_auto_id = 1

    # Chrome options
    options = uc.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--start-maximized")

    driver = uc.Chrome(options=options)
    driver.set_page_load_timeout(240)

    # Load dữ liệu cũ nếu có
    try:
        with open(output_file, "r", encoding="utf-8") as f:
            existing_data = json.load(f)
            all_data.extend(existing_data)
            if existing_data:
                current_auto_id = max(item.get("auto_id", 0) for item in existing_data) + 1
    except FileNotFoundError:
        pass

    # Mở trang đầu tiên
    url = "https://luatvietnam.vn/van-ban-luat-viet-nam.html?OrderBy=0&keywords=&lFieldId=46%2C11%2C57%2C45%2C25%2C35%2C23%2C52%2C15%2C194%2C65%2C3%2C61%2C1%2C54%2C62%2C24%2C5%2C28%2C21%2C34%2C27%2C66%2C64%2C40%2C59%2C18%2C7%2C13%2C87%2C31%2C29%2C2%2C86%2C55%2C88%2C49%2C4%2C51%2C48%2C85%2C89%2C26%2C19%2C42%2C6%2C12%2C22&EffectStatusId=0&DocTypeId=10&OrganId=0&page=1&pSize=20&ShowSapo=1"
    driver.get(url)

    for page_num in range(start_page, max_pages + 1):
        print(f"Đang xử lý trang {page_num}...")

        # Chờ danh sách hiện
        WebDriverWait(driver, 30).until(
            EC.presence_of_all_elements_located((By.XPATH, '//h2[@class="doc-title"]|//h3[@class="doc-title"]'))
        )

        # Lặp qua từng link
        idx = 0
        while True:
            try:
                law_links = driver.find_elements(By.XPATH, '//h2[@class="doc-title"]|//h3[@class="doc-title"]')
                if idx >= len(law_links):
                    break  # hết link trên trang này

                link = law_links[idx]
                a_tag = link.find_element(By.XPATH, './a')
                href = a_tag.get_attribute("href")
                idx += 1

                if not href:
                    continue

                # Mở chi tiết văn bản
                retry = 0
                while retry < 3:
                    try:
                        driver.get(href)
                        WebDriverWait(driver, 30).until(
                            EC.presence_of_element_located((By.XPATH, '//div[@class="the-document-body ndthaydoi noidungtracuu"]'))
                        )
                        break
                    except Exception:
                        retry += 1
                        print(f"Retry load {href} lần {retry}")
                        time.sleep(2)

                # Nếu sau 3 lần vẫn fail thì bỏ qua
                if retry == 3:
                    print(f"Bỏ qua link: {href}")
                    driver.back()
                    continue

                # Parse nội dung
                data_item = {}
                container = driver.find_element(By.XPATH, '//div[@class="the-document-body ndthaydoi noidungtracuu"]')
                all_elements = container.find_elements(By.XPATH, 'div[@class="docitem-1"]|div[@class="docitem-2"]|div[@class="docitem-3"]|div[@class="docitem-4"]|div[@class="docitem-5"]|div[@class="docitem-6"]|div[@class="docitem-7"]|div[@class="docitem-8"]|div[@class="docitem-9"]|div[@class="docitem-10"]|div[@class="docitem-11"]|div[@class="docitem-12"]|div[@class="docitem-13"]|div[@class="docitem-14"]|div[@class="docitem-15"]|div[@class="docitem-16"]|div[@class="docitem-17"]|div[@class="docitem-18"]')
                content_list = []

                for el in all_elements:
                    text = el.text.strip()
                    if not text:
                        continue
                    match = re.search(r"(?:Số:\s*)?(\d{1,3}/\d{4}/[A-Z0-9\-]+)", text)
                    if match:
                        data_item["law_id"] = match.group(1).strip()
                    else:
                        content_list.append(text)

                data_item["content"] = content_list  # mỗi p/table 1 dòng
                data_item["auto_id"] = current_auto_id
                current_auto_id += 1

                all_data.append(data_item)

                # Lưu sau mỗi mục
                with open(output_file, "w", encoding="utf-8") as f:
                    json.dump(all_data, f, ensure_ascii=False, indent=4)

                print(f"Đã xử lý xong văn bản {data_item.get('law_id', 'N/A')} (auto_id={data_item['auto_id']})")

                # Quay lại danh sách
                driver.back()
                WebDriverWait(driver, 30).until(
                    EC.presence_of_all_elements_located((By.XPATH, '//h2[@class="doc-title"]|//h3[@class="doc-title"]'))
                )

            except Exception as e:
                print(f"Lỗi tại mục {idx} trên trang {page_num}: {e}")
                try:
                    driver.back()
                except:
                    pass
                idx += 1

        # Sang trang tiếp theo
        try:
            next_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, '//a[text()="»"]'))
            )
            driver.execute_script("arguments[0].click();", next_button)
        except:
            print("Không còn trang tiếp theo.")
            break

    print("Hoàn tất cào dữ liệu!")
    driver.quit()



In [11]:
scrape_laws()

Đang xử lý trang 1...
Đã xử lý xong văn bản 58/2024/QH15 (auto_id=1)
Đã xử lý xong văn bản 92/2015/QH13 (auto_id=2)
Đã xử lý xong văn bản 29/2013/QH13 (auto_id=3)
Đã xử lý xong văn bản 18/2008/QH12 (auto_id=4)
Đã xử lý xong văn bản 10/2022/QH15 (auto_id=5)
Đã xử lý xong văn bản 130/2020/QH14 (auto_id=6)
Đã xử lý xong văn bản 13/2023/N (auto_id=7)
Đã xử lý xong văn bản 64/2025/QH15 (auto_id=8)
Đã xử lý xong văn bản 190/2025/QH15 (auto_id=9)
Đã xử lý xong văn bản 11/2022/QH15 (auto_id=10)
Đã xử lý xong văn bản 32/2024/QH15 (auto_id=11)
Đã xử lý xong văn bản 06/2017/QH14 (auto_id=12)
Đã xử lý xong văn bản 22/2018/QH14 (auto_id=13)
Đã xử lý xong văn bản 101/2015/QH13 (auto_id=14)
Đã xử lý xong văn bản 85/2015/QH13 (auto_id=15)
Đã xử lý xong văn bản 100/2015/QH13 (auto_id=16)
Đã xử lý xong văn bản 83/2015/QH13 (auto_id=17)
Đã xử lý xong văn bản 05/2007/QH12 (auto_id=18)
Đã xử lý xong văn bản 24/2008/QH12 (auto_id=19)
Đã xử lý xong văn bản 63/2014/QH13 (auto_id=20)
Đang xử lý trang 2...
Đã x

In [ ]:
import re
import time
import json
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

# Hàm khởi tạo driver mới
def init_driver():
    options = uc.ChromeOptions()
    options.headless = True  # chạy ẩn
    driver = uc.Chrome(options=options)
    driver.set_page_load_timeout(240)
    return driver

driver = init_driver()

base_url ="https://luatvietnam.vn/van-ban-luat-viet-nam.html?OrderBy=0&keywords=&lFieldId=46%2C11%2C57%2C45%2C25%2C35%2C23%2C52%2C15%2C194%2C65%2C3%2C61%2C1%2C54%2C62%2C24%2C5%2C28%2C21%2C34%2C27%2C66%2C64%2C40%2C59%2C18%2C7%2C13%2C87%2C31%2C29%2C2%2C86%2C55%2C88%2C49%2C4%2C51%2C48%2C85%2C89%2C26%2C19%2C42%2C6%2C12%2C22&EffectStatusId=0&DocTypeId=10&OrganId=0&page=1&pSize=20&ShowSapo=1"
current_auto_id = 1
all_data = []

try:
    for page_num in range(1, 25):  # ví dụ cào 5 trang đầu
        print(f"📄 Đang xử lý trang {page_num}...")

        # load trang danh sách
        driver.get(f"{base_url}&page={page_num}&pSize=20&ShowSapo=1")
        time.sleep(3)

        # lấy danh sách link văn bản
        law_links = driver.find_elements(By.XPATH, './/h2[@class="doc-title"]|//h3[@class="doc-title"]')
        urls = [link.find_element(By.XPATH, './a').get_attribute("href") for link in law_links]

        for idx, href in enumerate(urls, 1):
            try:
                driver.get(href)
                time.sleep(2)

                # parse nội dung văn bản
                data_item = {}
                container = driver.find_element(By.XPATH, '//div[@class="the-document-body ndthaydoi noidungtracuu"]')
                all_elements = container.find_elements(
                    By.XPATH,
                    'div[@class="docitem-1"]|div[@class="docitem-2"]|div[@class="docitem-3"]|div[@class="docitem-4"]|'
                    'div[@class="docitem-5"]|div[@class="docitem-6"]|div[@class="docitem-7"]|div[@class="docitem-8"]|'
                    'div[@class="docitem-9"]|div[@class="docitem-10"]|div[@class="docitem-11"]|div[@class="docitem-12"]|'
                    'div[@class="docitem-13"]|div[@class="docitem-14"]|div[@class="docitem-15"]|div[@class="docitem-16"]|'
                    'div[@class="docitem-17"]|div[@class="docitem-18"]'
                )

                content_list = []
                for el in all_elements:
                    text = el.text.strip()
                    if not text:
                        continue
                    match = re.search(r"(?:Số:\s*)?(\d{1,3}/\d{4}/[A-Z0-9\-]+)", text)
                    if match:
                        data_item["law_id"] = match.group(1).strip()
                    else:
                        content_list.append(text)

                data_item["content"] = " ".join(content_list)  # gộp thành đoạn
                data_item["auto_id"] = current_auto_id
                current_auto_id += 1

                all_data.append(data_item)
                print(f"✅ Đã cào {idx}/{len(urls)} văn bản ở trang {page_num}")

            except Exception as e:
                print(f"⚠️ Lỗi khi cào link {href}: {e}")
                continue

        # 👉 Sau mỗi 2 trang thì restart driver để tránh lỗi session
        if page_num % 2 == 0:
            driver.quit()
            driver = init_driver()
    print("Found links:", len(law_links))


finally:
    driver.quit()

# Xuất thử ra console
for item in all_data[:3]:
    print(item)

# 👉 Xuất ra file JSON
output_file = "law_data.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=4)

print(f"💾 Đã lưu {len(all_data)} văn bản vào {output_file}")


📄 Đang xử lý trang 1...
📄 Đang xử lý trang 2...
📄 Đang xử lý trang 3...
📄 Đang xử lý trang 4...
📄 Đang xử lý trang 5...
📄 Đang xử lý trang 6...
📄 Đang xử lý trang 7...
📄 Đang xử lý trang 8...
📄 Đang xử lý trang 9...
📄 Đang xử lý trang 10...
📄 Đang xử lý trang 11...
📄 Đang xử lý trang 12...
📄 Đang xử lý trang 13...
📄 Đang xử lý trang 14...
📄 Đang xử lý trang 15...
📄 Đang xử lý trang 16...
📄 Đang xử lý trang 17...
📄 Đang xử lý trang 18...
📄 Đang xử lý trang 19...
📄 Đang xử lý trang 20...
📄 Đang xử lý trang 21...
📄 Đang xử lý trang 22...
📄 Đang xử lý trang 23...
📄 Đang xử lý trang 24...
💾 Đã lưu 0 văn bản vào law_data.json
